### 1. Import Library

In [ ]:
!pip install -q segmentation-models-pytorch albumentations scikit-image

# Check for Tesla P100 GPU and show compatibility warning/instructions
import torch
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"Using GPU: {device_name}")
    if "P100" in device_name:
        print("\n" + "="*80)
        print("WARNING: Tesla P100 (compute capability 6.0) is not supported by the default PyTorch version on Kaggle.")
        print("This will cause the error: 'CUDA error: no kernel image is available for execution'.")
        print("\nTO FIX THIS, EITHER:")
        print("1. [Recommended] Change Kaggle Accelerator settings from 'GPU P100' to 'GPU T4 x2' (faster & fully compatible).")
        print("2. Or uncomment and run the next cell to install a compatible PyTorch version for P100.")
        print("="*80 + "\n")

In [ ]:
# RUN THIS CELL ONLY IF YOU ARE USING TESLA P100 AND WANT TO DOWNGRADE PYTORCH FOR COMPATIBILITY:
# !pip uninstall -y torch torchvision torchaudio
# !pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118


In [ ]:
import gdown
import zipfile
import os
import shutil
import random
import time
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
import torchvision.transforms.functional as TF

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

from collections import OrderedDict
import copy

### 2. Dataset Preparation

Download dataset from Google Drive

In [ ]:
file_id = '1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo'
url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'dataset.zip'

print("Downloading dataset from Google Drive...")
gdown.download(url, output_zip, quiet=False)

Extract Dataset

In [ ]:
extract_dir = 'dataset'
print("\nExtracting dataset...")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

### 3. Enhanced Class Imbalance Analysis

Ground truth masks have an extreme class imbalance (lesion vs background). This section performs an in-depth analysis:
- Overall pixel ratio (lesion vs background)
- Per-image positive pixel ratio distribution (histogram)
- Count of images with ZERO lesion pixels vs non-zero
- Box plot of positive pixel ratios across the dataset

In [ ]:
def analyze_class_imbalance_enhanced(base_dir):
    mask_dir = os.path.join(base_dir, 'train', 'masks')
    mask_files = sorted(glob.glob(os.path.join(mask_dir, '*')))

    total_pixels = 0
    total_white_pixels = 0
    per_image_ratios = []
    zero_lesion_count = 0
    nonzero_lesion_count = 0

    print(f"Analyzing {len(mask_files)} masks in train set...")
    for mask_path in mask_files:
        img = Image.open(mask_path).convert('L')
        arr = np.array(img)
        binary = (arr > 127).astype(np.uint8) if arr.max() > 1 else (arr > 0.5).astype(np.uint8)
        total_pixels += binary.size
        lesion_px = np.sum(binary)
        total_white_pixels += lesion_px
        ratio = lesion_px / binary.size * 100
        per_image_ratios.append(ratio)
        if lesion_px == 0:
            zero_lesion_count += 1
        else:
            nonzero_lesion_count += 1

    white_percent = (total_white_pixels / total_pixels) * 100
    bg_percent = 100.0 - white_percent

    print(f"\n========== ENHANCED IMBALANCE ANALYSIS ==========")
    print(f"Total Pixels: {total_pixels:,}")
    print(f"Lesion Pixels: {total_white_pixels:,} ({white_percent:.4f}%)")
    print(f"Background Pixels: {total_pixels - total_white_pixels:,} ({bg_percent:.4f}%)")
    print(f"\nPer-Image Statistics:")
    print(f"  Images with ZERO lesion pixels: {zero_lesion_count} ({zero_lesion_count/len(mask_files)*100:.1f}%)")
    print(f"  Images with NON-ZERO lesion pixels: {nonzero_lesion_count} ({nonzero_lesion_count/len(mask_files)*100:.1f}%)")
    ratios_arr = np.array(per_image_ratios)
    nonzero_ratios = ratios_arr[ratios_arr > 0]
    if len(nonzero_ratios) > 0:
        print(f"  Mean lesion ratio (all images): {ratios_arr.mean():.4f}%")
        print(f"  Mean lesion ratio (non-zero only): {nonzero_ratios.mean():.4f}%")
        print(f"  Median lesion ratio (non-zero): {np.median(nonzero_ratios):.4f}%")
        print(f"  Max lesion ratio: {ratios_arr.max():.4f}%")
        print(f"  Std lesion ratio: {ratios_arr.std():.4f}%")

    # --- Plot 1: Overall Class Distribution ---
    plt.figure(figsize=(18, 5))
    plt.subplot(1, 3, 1)
    classes = ['Background', 'Lesion (Stroke)']
    percentages = [bg_percent, white_percent]
    bars = plt.bar(classes, percentages, color=['#34495e', '#e74c3c'])
    plt.ylabel('Percentage (%)')
    plt.title('Overall Class Distribution (Pixels)')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
    plt.ylim(0, 110)

    # --- Plot 2: Per-Image Positive Pixel Ratio Histogram ---
    plt.subplot(1, 3, 2)
    plt.hist(ratios_arr, bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
    plt.axvline(x=ratios_arr.mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {ratios_arr.mean():.3f}%')
    if len(nonzero_ratios) > 0:
        plt.axvline(x=np.median(nonzero_ratios), color='green', linestyle=':', linewidth=2, label=f'Median (non-zero): {np.median(nonzero_ratios):.3f}%')
    plt.xlabel('Lesion Pixel Ratio (%)')
    plt.ylabel('Frequency')
    plt.title('Per-Image Lesion Ratio Distribution')
    plt.legend()
    plt.yscale('log')

    # --- Plot 3: Box Plot of Positive Pixel Ratios ---
    plt.subplot(1, 3, 3)
    box_data = [ratios_arr, nonzero_ratios] if len(nonzero_ratios) > 0 else [ratios_arr]
    box_labels = ['All images', 'Non-zero only'] if len(nonzero_ratios) > 0 else ['All images']
    bp = plt.boxplot(box_data, labels=box_labels, patch_artist=True)
    colors_box = ['#3498db', '#e74c3c']
    for patch, color in zip(bp['boxes'], colors_box[:len(box_data)]):
        patch.set_facecolor(color)
    plt.ylabel('Lesion Pixel Ratio (%)')
    plt.title('Box Plot of Lesion Ratios')
    plt.yscale('log')

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*50}")
    print(f"Recommendation: Use pos_weight = {bg_percent/white_percent:.1f}")
    print(f"for weighted BCE loss to balance gradient contribution.")
    print(f"Additionally, oversample positive images during training.")
    print(f"{'='*50}")

analyze_class_imbalance_enhanced('dataset')

### 3.b. Setup Model Weight Persistence

Set up directories for saving model weights. On Kaggle:
- `/kaggle/working/` = temporary (deleted after session)
- Kaggle Datasets = persistent across sessions

You can download weights from Kaggle Output or use Kaggle Dataset API.
The `MODELS_DIR` path is used throughout this notebook for saving/loading weights.

In [ ]:
import os
import zipfile
from pathlib import Path

# Create models directory
MODELS_DIR = 'trained_models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Flag for Kaggle environment
IS_KAGGLE = '/kaggle' in os.getcwd() or os.path.exists('/kaggle')

if IS_KAGGLE:
    print(f"Kaggle environment detected - weights will be saved to {MODELS_DIR}/")
    print("To persist weights across sessions, download the zip after training completes.")
else:
    print(f"Local environment - weights will be saved to {MODELS_DIR}/")


### 4. BalancedNCCTDataset with Online Patch Extraction & Weight Maps

This enhanced dataset extends the V5 concept with:
- **Online Patch Extraction**: For images WITH lesions, extract 256x256 patches centered on/around lesion regions. For images WITHOUT lesions, extract random 256x256 patches anywhere.
- **Positive Oversampling**: Internal list of 'positive' image indices ensures balanced batches.
- **Per-Pixel Class Weights**: Returns a `weight_map` where lesion pixels = (bg_pixels / lesion_pixels).
- **Enhanced Augmentations**: BrightnessContrast, RandomGamma, ElasticTransform, CoarseDropout.

In [ ]:
class NCCTDataset(Dataset):
    """Original V5 dataset - kept for validation/test (no augmentations for val)."""
    def __init__(self, split_dir, limit_size=None, transform=None):
        self.split_dir = split_dir
        self.transform = transform

        img_dir = os.path.join(split_dir, 'images')
        mask_dir = os.path.join(split_dir, 'masks')

        self.image_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)])
        self.mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])

        if limit_size is not None:
            self.image_paths = self.image_paths[:limit_size]
            self.mask_paths = self.mask_paths[:limit_size]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_pil = Image.open(self.image_paths[idx]).convert("L")
        mask_pil = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            if hasattr(self.transform, 'processors') or 'Compose' in str(type(self.transform)):
                # Albumentations
                img_np = np.array(img_pil)
                mask_np = np.array(mask_pil)
                augmented = self.transform(image=img_np, mask=mask_np)
                img = augmented['image']
                mask = augmented['mask']
            else:
                # Torchvision fallback
                img = img_pil
                mask = mask_pil
                if self.split_dir.endswith('train'):
                    if random.random() > 0.5:
                        img = TF.hflip(img)
                        mask = TF.hflip(mask)
                    if random.random() > 0.5:
                        img = TF.vflip(img)
                        mask = TF.vflip(mask)
                    angle = random.uniform(-15, 15)
                    img = TF.rotate(img, angle)
                    mask = TF.rotate(mask, angle)
                img = self.transform(img)
                mask = self.transform(mask)
        else:
            img = TF.to_tensor(img_pil)
            mask = TF.to_tensor(mask_pil)

        # Binarize mask to 0.0 or 1.0 (Fixes bilinear interpolation bug)
        if isinstance(mask, torch.Tensor):
            mask = mask.float()
            if mask.max() > 1.0:
                mask = (mask > 127.0).float()
            else:
                mask = (mask > 0.5).float()
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            mask_np = np.array(mask)
            if mask_np.max() > 1.0:
                mask = torch.tensor((mask_np > 127.0).astype(np.float32)).unsqueeze(0)
            else:
                mask = torch.tensor((mask_np > 0.5).astype(np.float32)).unsqueeze(0)

        label = 1.0 if torch.max(mask) > 0 else 0.0

        return img, mask, torch.tensor([label], dtype=torch.float32)

In [ ]:
class BalancedNCCTDataset(Dataset):
    """
    Enhanced dataset with:
    - Online patch extraction around lesion regions for positive samples
    - Random patch extraction for negative samples
    - Per-pixel class weight maps for gradient balancing
    - Enhanced augmentations
    """
    def __init__(self, split_dir, patch_size=256, transform=None, limit_size=None):
        self.split_dir = split_dir
        self.patch_size = patch_size
        self.transform = transform

        img_dir = os.path.join(split_dir, 'images')
        mask_dir = os.path.join(split_dir, 'masks')

        self.image_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)])
        self.mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])

        if limit_size is not None:
            self.image_paths = self.image_paths[:limit_size]
            self.mask_paths = self.mask_paths[:limit_size]

        # Pre-compute which images have lesions (positive samples)
        self.positive_indices = []
        self.negative_indices = []
        print("Pre-computing positive/negative sample indices...")
        for idx in range(len(self.image_paths)):
            mask_pil = Image.open(self.mask_paths[idx]).convert('L')
            arr = np.array(mask_pil)
            binary = (arr > 127).astype(np.uint8) if arr.max() > 1 else (arr > 0.5).astype(np.uint8)
            if binary.sum() > 0:
                self.positive_indices.append(idx)
            else:
                self.negative_indices.append(idx)
        print(f"  Positive samples: {len(self.positive_indices)}")
        print(f"  Negative samples: {len(self.negative_indices)}")

        # Pre-compute class weights for gradient balancing
        self.total_bg = 0
        self.total_lesion = 0
        for idx in range(len(self.image_paths)):
            mask_pil = Image.open(self.mask_paths[idx]).convert('L')
            arr = np.array(mask_pil)
            binary = (arr > 127).astype(np.uint8) if arr.max() > 1 else (arr > 0.5).astype(np.uint8)
            self.total_bg += binary.size - binary.sum()
            self.total_lesion += binary.sum()
        self.bg_weight = 1.0
        self.lesion_weight = self.total_bg / max(self.total_lesion, 1)
        print(f"  Class weight ratio (bg:lesion) = 1 : {self.lesion_weight:.1f}")

    def _get_random_patch_around_lesion(self, img_np, mask_np):
        """Extract a patch centered on/around a lesion region.
        Retries if patch has no lesion pixels (up to 5 attempts).
        """
        h, w = img_np.shape[:2]
        ps = self.patch_size

        # Find lesion pixels
        binary = (mask_np > 127).astype(np.uint8) if mask_np.max() > 1 else (mask_np > 0.5).astype(np.uint8)
        lesion_coords = np.argwhere(binary)

        # Retry up to 5 times to get a patch WITH lesion pixels
        best_patch = None
        best_lesion_count = 0
        for attempt in range(5):
            if len(lesion_coords) > 0:
                # Pick a random lesion pixel as center
                cy, cx = lesion_coords[random.randint(0, len(lesion_coords) - 1)]
                # Add random offset (grows per attempt to diversify)
                max_offset = min(10 + attempt * 20, 60)
                offset_y = random.randint(-max_offset, max_offset)
                offset_x = random.randint(-max_offset, max_offset)
                cy = max(0, min(h - 1, cy + offset_y))
                cx = max(0, min(w - 1, cx + offset_x))

                # Compute patch boundaries
                y_start = max(0, cy - ps // 2)
                x_start = max(0, cx - ps // 2)
                if y_start + ps > h:
                    y_start = h - ps
                if x_start + ps > w:
                    x_start = w - ps
                y, x = y_start, x_start
            else:
                # No lesion in image - random patch
                y = random.randint(0, max(0, h - ps))
                x = random.randint(0, max(0, w - ps))

            img_patch = img_np[y:y+ps, x:x+ps]
            mask_patch = mask_np[y:y+ps, x:x+ps]

            # Check lesion content
            patch_binary = (mask_patch > 127).astype(np.uint8) if mask_patch.max() > 1 else (mask_patch > 0.5).astype(np.uint8)
            lesion_count = patch_binary.sum()

            if lesion_count > best_lesion_count:
                best_patch = (img_patch, mask_patch)
                best_lesion_count = lesion_count

            # Accept if at least some lesion pixels found
            if lesion_count > 10:
                return img_patch, mask_patch

        # Return best attempt (will have lesion pixels if image had them)
        return best_patch if best_patch else (img_patch, mask_patch)

    def _compute_weight_map(self, mask):
        """Compute per-pixel class weights for gradient balancing."""
        if isinstance(mask, torch.Tensor):
            weight_map = torch.ones_like(mask)
            weight_map[mask > 0.5] = self.lesion_weight
        else:
            weight_map = np.ones_like(mask, dtype=np.float32)
            binary = (mask > 127).astype(np.uint8) if mask.max() > 1 else (mask > 0.5).astype(np.uint8)
            weight_map[binary > 0] = self.lesion_weight
        return weight_map

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_pil = Image.open(self.image_paths[idx]).convert("L")
        mask_pil = Image.open(self.mask_paths[idx]).convert("L")
        img_np = np.array(img_pil)
        mask_np = np.array(mask_pil)

        # Online patch extraction
        if self.transform is not None and 'train' in str(self.split_dir):
            img_patch, mask_patch = self._get_random_patch_around_lesion(img_np, mask_np)
        else:
            # For val/test, resize to patch_size
            img_patch = np.array(Image.fromarray(img_np).resize((self.patch_size, self.patch_size), Image.BILINEAR))
            mask_patch = np.array(Image.fromarray(mask_np).resize((self.patch_size, self.patch_size), Image.NEAREST))

        # Compute weight map before augmentation (on original mask patch)
        weight_map = self._compute_weight_map(mask_patch)

        if self.transform:
            augmented = self.transform(image=img_patch, mask=mask_patch, weight_map=weight_map)
            img = augmented['image']
            mask = augmented['mask']
            if 'weight_map' in augmented:
                weight_map = augmented['weight_map']
        else:
            img = TF.to_tensor(img_patch)
            mask = TF.to_tensor(mask_patch)
            weight_map = torch.tensor(weight_map).unsqueeze(0).float()

        # Binarize mask
        if isinstance(mask, torch.Tensor):
            mask = mask.float()
            if mask.max() > 1.0:
                mask = (mask > 127.0).float()
            else:
                mask = (mask > 0.5).float()
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            mask_np_final = np.array(mask)
            if mask_np_final.max() > 1.0:
                mask = torch.tensor((mask_np_final > 127.0).astype(np.float32)).unsqueeze(0)
            else:
                mask = torch.tensor((mask_np_final > 0.5).astype(np.float32)).unsqueeze(0)

        if isinstance(weight_map, np.ndarray):
            weight_map = torch.tensor(weight_map).float()
        if weight_map.ndim == 2:
            weight_map = weight_map.unsqueeze(0)

        label = 1.0 if torch.max(mask) > 0 else 0.0
        return img, mask, weight_map, torch.tensor([label], dtype=torch.float32)

Define train/val transforms with enhanced augmentations

In [ ]:
def get_enhanced_transforms():
    """
    Enhanced augmentations for training:
    - Keep V5: Resize(256,256), HorizontalFlip, VerticalFlip, ShiftScaleRotate, Normalize, ToTensorV2
    - ADD: RandomBrightnessContrast, RandomGamma, ElasticTransform, CoarseDropout
    """
    train_transform = A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.ElasticTransform(alpha=1, sigma=50, p=0.3),
        A.CoarseDropout(p=0.3),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ], additional_targets={'weight_map': 'image'})

    val_transform = A.Compose([
        A.Resize(256, 256),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])

    return train_transform, val_transform

In [ ]:
def get_balanced_dataloaders(base_dir, batch_size=8, limit_size=None):
    """
    Create dataloaders with BalancedNCCTDataset for train
    and regular NCCTDataset for val/test.
    """
    train_transform, val_transform = get_enhanced_transforms()

    train_dataset = BalancedNCCTDataset(
        os.path.join(base_dir, 'train'),
        transform=train_transform,
        limit_size=limit_size
    )
    val_dataset = NCCTDataset(
        os.path.join(base_dir, 'val'),
        transform=val_transform,
        limit_size=limit_size
    )
    test_dataset = NCCTDataset(
        os.path.join(base_dir, 'test'),
        transform=val_transform,
        limit_size=limit_size
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, train_dataset

### 5. Advanced Loss Functions

Three novel loss classes designed for extreme class imbalance:

- **FocalTverskyLoss**: Focal modulation of Tversky index (gamma=4/3). Focuses more on false positives (alpha=0.7, beta=0.3).
- **AsymmetricUnifiedFocalLoss**: Combines asymmetric Focal Loss with Focal Tversky for handling extreme imbalance.
- **WeightedComboLoss**: Weighted BCE (with dynamic pos_weight) + FocalTverskyLoss + Lovász-Hinge loss (optimizes IoU).

In [ ]:
class FocalTverskyLoss(nn.Module):
    """
    Focal Tversky Loss
    - alpha=0.7, beta=0.3: Focus MORE on false positives (improves precision)
    - gamma=4/3: Focal modulation (focus on hard cases)
    Formula: FT(p,t) = (1 - Tversky)^(1/gamma)
    """
    def __init__(self, alpha=0.7, beta=0.3, gamma=4/3, epsilon=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.epsilon = epsilon

    def forward(self, inputs, targets, weight_map=None):
        inputs_sigmoid = torch.sigmoid(inputs)
        targets = (targets > 0.5).float()

        # Flatten if needed
        if inputs_sigmoid.dim() > 2:
            inputs_sigmoid = inputs_sigmoid.view(inputs_sigmoid.size(0), -1)
            targets = targets.view(targets.size(0), -1)

        intersection = (inputs_sigmoid * targets).sum(dim=1)
        fps = (inputs_sigmoid * (1.0 - targets)).sum(dim=1)
        fns = ((1.0 - inputs_sigmoid) * targets).sum(dim=1)

        tversky = (intersection + self.epsilon) / (
            intersection + self.alpha * fps + self.beta * fns + self.epsilon
        )

        focal_tversky = (1.0 - tversky) ** (1.0 / self.gamma)
        return focal_tversky.mean()

In [ ]:
class AsymmetricUnifiedFocalLoss(nn.Module):
    """
    Asymmetric Unified Focal Loss
    Combines asymmetric Focal Loss with Focal Tversky
    - delta=0.6: Controls class asymmetry
    - gamma=0.8: Focal parameter
    Designed for extreme class imbalance scenarios.
    """
    def __init__(self, delta=0.6, gamma=0.8, epsilon=1e-6):
        super().__init__()
        self.delta = delta
        self.gamma = gamma
        self.epsilon = epsilon

    def forward(self, inputs, targets, weight_map=None):
        targets = (targets > 0.5).float()

        # --- Asymmetric Focal Loss Component ---
        inputs_sigmoid = torch.sigmoid(inputs)
        pt = inputs_sigmoid * targets + (1 - inputs_sigmoid) * (1 - targets)
        focal_weight = (1 - pt) ** self.gamma

        # Asymmetric focusing: weight positive samples more
        asymmetric_weight = self.delta * targets + (1 - self.delta) * (1 - targets)

        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        if weight_map is not None:
            bce = bce * weight_map
        focal_loss = (asymmetric_weight * focal_weight * bce).mean()

        # --- Focal Tversky Component ---
        flat_pred = inputs_sigmoid.view(inputs_sigmoid.size(0), -1)
        flat_targets = targets.view(targets.size(0), -1)

        tp = (flat_pred * flat_targets).sum(dim=1)
        fp = (flat_pred * (1.0 - flat_targets)).sum(dim=1)
        fn = ((1.0 - flat_pred) * flat_targets).sum(dim=1)

        tversky = (tp + self.epsilon) / (tp + self.delta * fp + (1 - self.delta) * fn + self.epsilon)
        focal_tversky = (1.0 - tversky) ** (1.0 / self.gamma)

        return focal_loss + focal_tversky.mean()

In [ ]:
def lovasz_grad(gt_sorted):
    """
    Computes gradient of the Lovasz extension w.r.t sorted errors.
    See Alg. 1 in "The Lovasz-Softmax Loss: A Tractable Surrogate for the Optimization of the Intersection-Over-Union Measure in Neural Networks"
    """
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted.float()).cumsum(0)
    jaccard = 1. - intersection / (union + 1e-12)
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits, labels):
    """Binary Lovasz hinge loss for flat inputs."""
    if len(labels) == 0:
        return logits.sum() * 0.
    signs = 2. * labels - 1.
    errors = 1. - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm = perm.data
    gt_sorted = labels[perm]
    grad = lovasz_grad(gt_sorted)
    loss = torch.dot(F.relu(errors_sorted), grad)
    return loss


def lovasz_hinge(logits, labels, per_image=True, ignore=None):
    """
    Binary Lovasz hinge loss.
      logits: [B, 1, H, W] Variable, logits at each pixel
      labels: [B, 1, H, W] Tensor, binary ground truth masks (0 or 1)
      per_image: compute the loss per image instead of per batch
    """
    if per_image:
        losses = []
        for logit, label in zip(logits, labels):
            logit_flat = logit.view(-1)
            label_flat = label.view(-1).float()
            loss = lovasz_hinge_flat(logit_flat, label_flat)
            losses.append(loss)
        loss = torch.mean(torch.stack(losses))
    else:
        loss = lovasz_hinge_flat(logits.view(-1), labels.view(-1).float())
    return loss

In [ ]:
class RobustBCEDiceLoss(nn.Module):
    """
    Robust BCE + Dice Loss — Stable combination proven for extreme imbalance.
    - Weighted BCE with pos_weight (no additional weight_map to avoid double-weighting)
    - Focal Tversky Loss (better gradient behavior than plain Dice)
    - NO Lovász-Hinge (can produce negative loss and destabilize training)
    - Weights: BCE=0.3, Tversky=0.7 focuses more on overlap quality
    """
    def __init__(self, bce_weight=0.3, tversky_weight=0.7, pos_weight=66.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.tversky_weight = tversky_weight
        self.pos_weight = pos_weight
        self.focal_tversky = FocalTverskyLoss(alpha=0.7, beta=0.3, gamma=4/3)

    def forward(self, inputs, targets):
        targets_bin = (targets > 0.5).float()

        # 1. Weighted BCE (class-balanced via pos_weight only)
        pw = torch.tensor([self.pos_weight], device=inputs.device)
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs, targets_bin, pos_weight=pw
        )

        # 2. Focal Tversky Loss (focuses on hard cases, stable gradients)
        tversky_loss = self.focal_tversky(inputs, targets_bin)

        return self.bce_weight * bce_loss + self.tversky_weight * tversky_loss
        return total

In [ ]:
def dice_coeff(pred, target, threshold=0.5, epsilon=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()
    target = (target > 0.5).float()
    inter = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2. * inter + epsilon) / (union + epsilon)
    return dice.mean().item()


def calculate_segmentation_metrics(pred_logits, targets, threshold=0.5, epsilon=1e-6):
    # Threshold predictions
    pred_1 = (torch.sigmoid(pred_logits) > threshold).float()
    target_1 = (targets > 0.5).float()

    # Class 1: Lesion / Stroke
    inter_1 = (pred_1 * target_1).sum(dim=(2, 3))
    union_1 = pred_1.sum(dim=(2, 3)) + target_1.sum(dim=(2, 3))
    dice_1 = (2. * inter_1 + epsilon) / (union_1 + epsilon)
    iou_1 = (inter_1 + epsilon) / (union_1 - inter_1 + epsilon)

    tp_1 = inter_1
    fp_1 = (pred_1 * (1.0 - target_1)).sum(dim=(2, 3))
    fn_1 = ((1.0 - pred_1) * target_1).sum(dim=(2, 3))
    prec_1 = (tp_1 + epsilon) / (tp_1 + fp_1 + epsilon)
    rec_1 = (tp_1 + epsilon) / (tp_1 + fn_1 + epsilon)

    # Class 0: Background
    pred_0 = 1.0 - pred_1
    target_0 = 1.0 - target_1
    inter_0 = (pred_0 * target_0).sum(dim=(2, 3))
    union_0 = pred_0.sum(dim=(2, 3)) + target_0.sum(dim=(2, 3))
    dice_0 = (2. * inter_0 + epsilon) / (union_0 + epsilon)
    iou_0 = (inter_0 + epsilon) / (union_0 - inter_0 + epsilon)

    tp_0 = inter_0
    fp_0 = (pred_0 * (1.0 - target_0)).sum(dim=(2, 3))
    fn_0 = ((1.0 - pred_0) * target_0).sum(dim=(2, 3))
    prec_0 = (tp_0 + epsilon) / (tp_0 + fp_0 + epsilon)
    rec_0 = (tp_0 + epsilon) / (tp_0 + fn_0 + epsilon)

    return {
        'lesion_dice': dice_1.mean().item(),
        'lesion_iou': iou_1.mean().item(),
        'lesion_precision': prec_1.mean().item(),
        'lesion_recall': rec_1.mean().item(),
        'background_dice': dice_0.mean().item(),
        'background_iou': iou_0.mean().item(),
        'background_precision': prec_0.mean().item(),
        'background_recall': rec_0.mean().item(),
    }

### 6. Models

All V5 models are kept unchanged. Two new models added:
- UNet_EffNet_B4 (efficientnet-b4 encoder, stronger than b3)
- DeepSupervisionUNet (ResNet34 encoder with auxiliary output heads at multiple decoder levels)

#### 6.a. CSPNet (NCCTSegmentationModel)

In [ ]:
class CSPBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = out_channels // 2

        self.main_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.skip_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.transition = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        main_out = self.main_conv(x)
        skip_out = self.skip_conv(x)
        merged = torch.cat([main_out, skip_out], dim=1)
        return self.transition(merged)

class NCCTSegmentationModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder (Backbone)
        self.initial_conv = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.layer1 = CSPBlock(32, 64)   # Output: 64 channels
        self.layer2 = CSPBlock(64, 128)  # Output: 128 channels
        self.layer3 = CSPBlock(128, 256) # Output: 256 channels

        self.pool = nn.MaxPool2d(2, 2)

        # Decoder with Skip Connections
        # features (256 channels) -> upconv1 (128 channels)
        self.upconv1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        # concat(up1:128, layer3:256) = 384
        self.decoder1 = CSPBlock(128 + 256, 128)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        # concat(up2:64, layer2:128) = 192
        self.decoder2 = CSPBlock(64 + 128, 64)

        self.upconv3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        # concat(up3:32, layer1:64) = 96
        self.decoder3 = CSPBlock(32 + 64, 32)

        self.seg_output = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        x0 = F.relu(self.initial_conv(x))

        x1 = self.layer1(x0)
        p1 = self.pool(x1)

        x2 = self.layer2(p1)
        p2 = self.pool(x2)

        x3 = self.layer3(p2)
        features = self.pool(x3)

        # Decoder
        d1 = self.upconv1(features)
        d1 = torch.cat([d1, x3], dim=1)
        d1 = self.decoder1(d1)

        d2 = self.upconv2(d1)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.decoder2(d2)

        d3 = self.upconv3(d2)
        d3 = torch.cat([d3, x1], dim=1)
        d3 = self.decoder3(d3)

        seg_out = self.seg_output(d3)
        return seg_out

#### 6.b. UNet (ResNet34)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )

    def forward(self, x):
        return self.model(x)

#### 6.c. Attention UNet (ResNet34 + scse)

In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            attention_type="scse",
            activation=None
        )

    def forward(self, x):
        return self.model(x)

#### 6.d. ResUNet (ResNet50)

In [ ]:
class ResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )

    def forward(self, x):
        return self.model(x)

#### 6.e. UNet_EffNet (EfficientNet-b3)
EfficientNet dikenal memiliki fitur ekstraksi yang sangat kuat dengan jumlah parameter yang relatif lebih sedikit, sehingga dapat menghasilkan akurasi yang lebih baik.

In [ ]:
class UNet_EffNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="efficientnet-b3",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)

#### 6.f. Linknet (ResNet34)

In [ ]:
class Linknet_ResNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Linknet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)

#### 6.g. FPN (ResNeXt50)

In [ ]:
class FPN_ResNext(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.FPN(
            encoder_name="resnext50_32x4d",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)

#### 6.h. PSPNet (ResNet50)

In [ ]:
class PSPNet_ResNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.PSPNet(
            encoder_name="resnet50",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)

#### 6.i. PAN (MobileNetV2)

In [ ]:
class PAN_MobileNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.PAN(
            encoder_name="mobilenet_v2",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)

#### 6.j. UNet_EffNet_B4 (NEW - Stronger EfficientNet encoder)
UNet with EfficientNet-b4 encoder (stronger than b3). Suitable for Kaggle T4 x2 GPU.

In [ ]:
class UNet_EffNet_B4(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="efficientnet-b4",
            encoder_weights="imagenet",
            in_channels=in_channels,
            classes=out_channels,
            activation=None
        )
    def forward(self, x):
        return self.model(x)

#### 6.k. DeepSupervisionUNet (EXPERIMENTAL - not used in training pipeline)\nUNet with ResNet34 encoder and auxiliary output heads at multiple decoder levels.\nNOTE: Custom decoder may have channel mismatch with some SMP encoder versions.\nTraining pipeline uses UNet_EffNet_B4 instead (stable smp.Unet wrapper).

In [ ]:
class DeepSupervisionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.encoder = smp.encoders.get_encoder(
            "resnet34",
            in_channels=in_channels,
            weights="imagenet"
        )

        # Decoder blocks
        # encoder channels: [64, 64, 128, 256, 512]
        self.dec4 = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.dec3 = nn.Sequential(
            nn.Conv2d(256 + 256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.dec2 = nn.Sequential(
            nn.Conv2d(128 + 128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.upsample = nn.UpsamplingBilinear2d(scale_factor=2)

        # Final segmentation head
        self.seg_head = nn.Conv2d(64, out_channels, kernel_size=1)

        # Auxiliary heads for deep supervision
        self.aux_head1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, out_channels, kernel_size=8, stride=4, padding=2, output_padding=0),
        )
        self.aux_head2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, out_channels, kernel_size=4, stride=2, padding=1, output_padding=0),
        )

    def forward(self, x):
        # Encoder features
        enc = self.encoder(x)
        # enc[0]: 64ch, H/2
        # enc[1]: 64ch, H/4
        # enc[2]: 128ch, H/8
        # enc[3]: 256ch, H/16
        # enc[4]: 512ch, H/32

        # Decoder level 4
        d4 = self.dec4(enc[4])  # 512 -> 256, H/32
        d4_up = self.upsample(d4)  # H/16

        # Decoder level 3
        d3_in = torch.cat([d4_up, enc[3]], dim=1)  # 256+256=512
        d3 = self.dec3(d3_in)  # 512 -> 128, H/16
        aux1_out = self.aux_head1(d3)  # Deep supervision output 1
        # Resize aux1 to match input
        aux1_out = F.interpolate(aux1_out, size=x.shape[2:], mode='bilinear', align_corners=False)
        d3_up = self.upsample(d3)  # H/8

        # Decoder level 2
        d2_in = torch.cat([d3_up, enc[2]], dim=1)  # 128+128=256
        d2 = self.dec2(d2_in)  # 256 -> 64, H/8
        aux2_out = self.aux_head2(d2)  # Deep supervision output 2
        aux2_out = F.interpolate(aux2_out, size=x.shape[2:], mode='bilinear', align_corners=False)
        d2_up = self.upsample(d2)  # H/4

        # Decoder level 1
        d1_in = torch.cat([d2_up, enc[1]], dim=1)  # 64+64=128
        d1 = self.dec1(d1_in)  # 128 -> 64, H/4
        d1_up = self.upsample(d1)  # H/2
        d1_up = self.upsample(d1_up)  # H/1

        # Final output
        final_out = self.seg_head(d1_up)

        if self.training:
            return [final_out, aux1_out, aux2_out]
        else:
            return final_out

#### 6.l. Bias Initialization for Imbalanced Segmentation

Critical fix for extreme class imbalance: initialize the final conv layer bias to bias initial predictions toward the lesion class.

Default sigmoid output at initialization: σ(0) = 0.5. With pos_weight ≈ 66, we want initial output ≈ log(66) so σ ≈ 0.985 (predicting lesion more often), preventing the model from immediately converging to "predict all background".

In [ ]:
def init_seg_head_bias(model, pos_weight=66.0, bias_value=None):
    """
    Initialize the bias of the final segmentation convolution layer
    to push initial predictions toward the lesion class.
    
    Bias = log(pos_weight) shifts initial sigmoid output from 0.5 to
    pos_weight/(1+pos_weight) ≈ 0.985 for lesion class.
    """
    if bias_value is None:
        bias_value = np.log(pos_weight)
    
    count = 0
    for name, module in model.named_modules():
        # Find the final segmentation head conv layer
        if isinstance(module, nn.Conv2d) and module.out_channels == 1 and module.kernel_size == (1, 1):
            if module.bias is not None:
                nn.init.constant_(module.bias, bias_value)
                print(f"  Initialized bias of {name} to {bias_value:.2f} (sig={torch.sigmoid(torch.tensor(bias_value)):.3f})")
                count += 1
            # Also initialize weights to small random values
            nn.init.normal_(module.weight, mean=0.0, std=0.001)
    
    if count == 0:
        print("  Warning: No 1x1 conv with out_channels=1 found. Trying alternative approach...")
        for name, param in model.named_parameters():
            if 'bias' in name and param.numel() == 1 and 'seg' in name.lower() or 'out' in name.lower() or 'head' in name.lower():
                nn.init.constant_(param, bias_value)
                print(f"  Initialized {name} to {bias_value:.2f}")
                break
    return model

print("✓ Bias initialization function defined.")
print("  Usage: model = init_seg_head_bias(model, pos_weight=66.6)")

### 7. Balanced Batch Sampler

Creates balanced batches ensuring at least 2-3 positive samples per batch.
- Identifies positive/negative samples from the dataset
- In each batch: includes positive samples + fills rest with negative samples
- If more positives than batch size, randomly subsamples

In [ ]:
class BalancedBatchSampler(Sampler):
    """
    Batch sampler that ensures at least min_pos positive samples per batch.
    - Uses pre-computed positive_indices from the dataset
    - In each batch: includes min_pos positive samples, fills rest with negative
    - Shuffles indices each epoch
    """
    def __init__(self, dataset, batch_size, min_pos=3, shuffle=True):
        super().__init__()
        self.dataset = dataset
        self.batch_size = batch_size
        self.min_pos = min(min_pos, batch_size)
        self.shuffle = shuffle

        # Get positive/negative indices from dataset
        if hasattr(dataset, 'positive_indices'):
            self.positive_indices = list(dataset.positive_indices)
            self.negative_indices = list(dataset.negative_indices)
        else:
            raise AttributeError("Dataset must have positive_indices and negative_indices attributes")

        self.n_pos = len(self.positive_indices)
        self.n_neg = len(self.negative_indices)
        print(f"BalancedBatchSampler: {self.n_pos} positive, {self.n_neg} negative samples")

    def __iter__(self):
        pos = self.positive_indices.copy()
        neg = self.negative_indices.copy()

        if self.shuffle:
            random.shuffle(pos)
            random.shuffle(neg)

        n_pos_per = self.min_pos
        n_neg_per = self.batch_size - n_pos_per

        pos_idx, neg_idx = 0, 0
        batches = []

        while pos_idx < self.n_pos or neg_idx < self.n_neg:
            batch = []

            # Add positive samples
            for _ in range(n_pos_per):
                if pos_idx < self.n_pos:
                    batch.append(pos[pos_idx])
                    pos_idx += 1

            # Fill with negative samples
            remaining = self.batch_size - len(batch)
            for _ in range(remaining):
                if neg_idx < self.n_neg:
                    batch.append(neg[neg_idx])
                    neg_idx += 1

            if len(batch) >= 2:  # Minimum batch size
                if self.shuffle:
                    random.shuffle(batch)
                batches.append(batch)

        return iter(batches)

    def __len__(self):
        # Estimate number of batches
        pos_batches = (self.n_pos + self.min_pos - 1) // self.min_pos
        return max(pos_batches, 1)

### 8. Enhanced Training Loop

Features:
- **Gradient Clipping**: max_norm=1.0 to prevent exploding gradients
- **Two-phase Learning Rate**: Phase 1 (epochs 1-15): lr=1e-4 cosine annealing. Phase 2 (epochs 16-50): lr=1e-5 with ReduceLROnPlateau.
- **Deep Supervision** (for DeepSupervisionUNet): Weighted sum of auxiliary losses
- **Weighted Loss Aggregation**: Uses weight_map from dataset
- **EMA (Exponential Moving Average)**: Maintains EMA weights, used for validation
- **Early Stopping**: patience=10, min_delta=0.001

In [ ]:
class EMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}

    def register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

In [ ]:
def train_model_weighted(model, train_loader, val_loader, epochs=50, model_name="Model",
                         patience=10, min_delta=0.001, use_ema=True):
    """
    Enhanced training loop with:
    - Two-phase learning rate
    - Gradient clipping (max_norm=1.0)
    - Deep supervision support
    - Weight map usage in loss
    - EMA weight averaging
    - Early stopping
    """
    # Two-phase optimizer setup
    optimizer_phase1 = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler_phase1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_phase1, T_max=15, eta_min=1e-5)

    optimizer_phase2 = torch.optim.Adam(model.parameters(), lr=1e-5)
    scheduler_phase2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_phase2, mode='min', factor=0.5, patience=5, min_lr=1e-7
    )

    # Loss function
    # Dynamically compute pos_weight from dataset if available
    ds = train_loader.dataset
    if hasattr(ds, 'lesion_weight'):
        pos_w = ds.lesion_weight
    else:
        pos_w = 66.0
    print(f"  Using pos_weight = {pos_w:.1f}")

    criterion = RobustBCEDiceLoss(pos_weight=pos_w).cuda()

    # EMA setup
    ema = EMA(model, decay=0.999) if use_ema else None
    if ema:
        ema.register()

    print(f"\nTraining {model_name} (Enhanced)...")
    print(f"  Two-phase LR: 1e-4 (cosine, 15 ep) -> 1e-5 (plateau, 35 ep)")
    print(f"  Gradient clipping: max_norm=1.0")
    print(f"  EMA: {use_ema}")
    print(f"  Deep supervision: {'yes' if hasattr(model, 'aux_head1') else 'no'}")
    print(f"{'Epoch':<8} | {'T-Loss':<8} | {'V-Loss':<8} | {'V-Dice':<8} | {'LR':<12} | {'Phase':<7} | {'Time':<8}")
    print("-" * 75)

    best_dice = 0
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_dice': []}

    def has_weight_map(batch):
        return len(batch) == 4  # img, mask, weight_map, label

    for epoch in range(epochs):
        start_time = time.time()

        # Determine phase
        if epoch < 15:
            phase = 1
            optimizer = optimizer_phase1
            scheduler = scheduler_phase1
        else:
            phase = 2
            optimizer = optimizer_phase2
            scheduler = scheduler_phase2

        # Training
        model.train()
        train_loss = 0

        for batch in train_loader:
            if has_weight_map(batch):
                images, masks, weight_maps, _ = batch
            else:
                images, masks, _ = batch
                weight_maps = None

            images, masks = images.cuda(), masks.cuda()
            if weight_maps is not None:
                weight_maps = weight_maps.cuda()

            optimizer.zero_grad()
            outputs = model(images)

            # Handle deep supervision
            if isinstance(outputs, list):
                # Deep supervision: weighted sum of auxiliary losses
                aux1, aux2, final_out = outputs[0], outputs[1], outputs[2]
                loss_final = criterion(final_out, masks)
                loss_aux1 = criterion(aux1, masks)
                loss_aux2 = criterion(aux2, masks)
                loss = 0.3 * loss_final + 0.4 * loss_aux1 + 0.3 * loss_aux2
            else:
                loss = criterion(outputs, masks)

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # EMA update
            if ema:
                ema.update()

            train_loss += loss.item()

        # Validation (with EMA if available)
        model.eval()
        if ema:
            ema.apply_shadow()

        val_loss, val_dice = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                if has_weight_map(batch):
                    images, masks, weight_maps, _ = batch
                    weight_maps = weight_maps.cuda()
                else:
                    images, masks, _ = batch
                    weight_maps = None

                images, masks = images.cuda(), masks.cuda()
                outputs = model(images)

                if isinstance(outputs, list):
                    outputs = outputs[0]  # Use final output for val

                loss = criterion(outputs, masks)
                val_loss += loss.item()
                val_dice += dice_coeff(outputs, masks)

        if ema:
            ema.restore()

        end_time = time.time()
        duration = end_time - start_time

        avg_t_loss = train_loss / len(train_loader)
        avg_v_loss = val_loss / len(val_loader)
        avg_v_dice = val_dice / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(avg_t_loss)
        history['val_loss'].append(avg_v_loss)
        history['val_dice'].append(avg_v_dice)

        # Phase 1: cosine annealing scheduler steps every epoch
        if phase == 1:
            scheduler.step()
        # Phase 2: ReduceLROnPlateau based on val_loss
        else:
            scheduler.step(avg_v_loss)

        print(f"{epoch+1:<8} | {avg_t_loss:<8.4f} | {avg_v_loss:<8.4f} | {avg_v_dice:<8.4f} | {current_lr:<12.8f} | {phase:<7} | {duration:<8.2f}s")

        if avg_v_dice > best_dice:
            best_dice = avg_v_dice
            if ema:
                ema.apply_shadow()
                save_path = os.path.join(MODELS_DIR, f"{model_name.lower().replace(' ', '_')}.pth")
                torch.save(model.state_dict(), save_path)
                ema.restore()
            else:
                save_path = os.path.join(MODELS_DIR, f"{model_name.lower().replace(' ', '_')}.pth")
                torch.save(model.state_dict(), save_path)
            print(f"  -> New best model saved! Val Dice: {avg_v_dice:.4f}")

        # Early stopping
        if avg_v_loss < best_val_loss - min_delta:
            best_val_loss = avg_v_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    # Load best model (with EMA if available)
    load_path = os.path.join(MODELS_DIR, f"{model_name.lower().replace(' ', '_')}.pth")
    if os.path.exists(load_path):
        model.load_state_dict(torch.load(load_path))
    else:
        print(f"  Warning: Best weights not found at {load_path}, using last checkpoint")
    return model, history

### 9. Post-Processing

Two key post-processing functions:
- `post_process_mask`: Connected components analysis to remove small lesion components (< min_lesion_area pixels)
- `apply_tta`: Test-Time Augmentation with horizontal + vertical flips (4 augmentations averaged)

In [ ]:
from skimage import measure


def post_process_mask(mask_tensor, min_lesion_area=50):
    """
    Apply connected components analysis to remove small lesion components.
    Args:
        mask_tensor: [1, H, W] or [H, W] binary torch tensor (values 0 or 1)
        min_lesion_area: minimum pixel area to keep
    Returns:
        Cleaned mask of same shape
    """
    if isinstance(mask_tensor, torch.Tensor):
        mask_np = mask_tensor.cpu().numpy()
    else:
        mask_np = mask_tensor

    if mask_np.ndim == 3:
        mask_np = mask_np.squeeze(0)

    binary = (mask_np > 0.5).astype(np.uint8)

    # Label connected components
    labeled = measure.label(binary, connectivity=2)
    props = measure.regionprops(labeled)

    # Create output mask
    cleaned = np.zeros_like(binary)

    for prop in props:
        if prop.area >= min_lesion_area:
            cleaned[labeled == prop.label] = 1

    return torch.from_numpy(cleaned).float().unsqueeze(0)


def apply_tta(model, image, n_tta=4):
    """
    Test-Time Augmentation.
    Applies horizontal + vertical flips, averages predictions.
    n_tta=4: (original, hflip, vflip, hflip+vflip)
    """
    model.eval()
    with torch.no_grad():
        preds = []

        # Original
        out = model(image)
        if isinstance(out, list):
            out = out[0]
        preds.append(torch.sigmoid(out))

        # Horizontal flip
        img_hflip = torch.flip(image, dims=[3])
        out = model(img_hflip)
        if isinstance(out, list):
            out = out[0]
        preds.append(torch.flip(torch.sigmoid(out), dims=[3]))

        if n_tta >= 4:
            # Vertical flip
            img_vflip = torch.flip(image, dims=[2])
            out = model(img_vflip)
            if isinstance(out, list):
                out = out[0]
            preds.append(torch.flip(torch.sigmoid(out), dims=[2]))

            # Both flips
            img_hvflip = torch.flip(image, dims=[2, 3])
            out = model(img_hvflip)
            if isinstance(out, list):
                out = out[0]
            preds.append(torch.flip(torch.sigmoid(out), dims=[2, 3]))

        # Average predictions
        mean_pred = torch.mean(torch.stack(preds), dim=0)

    return mean_pred

### 10. Advanced Evaluation with TTA + Post-Processing

Evaluates models using:
- Test-Time Augmentation (4 augmentations averaged)
- Post-processing to remove small artifacts (< 50 pixels)
- Same comprehensive metrics as V5: Dice, IoU, Precision, Recall per class

In [ ]:
def eval_model_advanced(models_dict, test_loader, use_tta=True, use_post_process=True, min_lesion_area=50):
    """
    Advanced evaluation with TTA and post-processing.
    """
    results = {}
    tta_str = "TTA" if use_tta else "No TTA"
    pp_str = "+PP" if use_post_process else ""

    print(f"\n{'='*100}")
    print(f"Evaluation Mode: {tta_str}{pp_str}")
    print(f"{'='*100}")
    print(f"{'Model':<20} | {'Class':<16} | {'Dice':<10} | {'IoU':<10} | {'Precision':<12} | {'Recall':<12}")
    print("-" * 90)

    for name, model in models_dict.items():
        model.eval()

        accumulated_metrics = {
            'lesion_dice': 0.0, 'lesion_iou': 0.0, 'lesion_precision': 0.0, 'lesion_recall': 0.0,
            'background_dice': 0.0, 'background_iou': 0.0, 'background_precision': 0.0, 'background_recall': 0.0
        }

        num_batches = len(test_loader)
        with torch.no_grad():
            for images, masks, _ in test_loader:
                images, masks = images.cuda(), masks.cuda()

                # TTA prediction
                if use_tta:
                    pred_probs = apply_tta(model, images)
                else:
                    out = model(images)
                    if isinstance(out, list):
                        out = out[0]
                    pred_probs = torch.sigmoid(out)

                # Post-processing
                if use_post_process:
                    pred_binary = (pred_probs > 0.5).float()
                    processed = []
                    for b in range(pred_binary.size(0)):
                        cleaned = post_process_mask(pred_binary[b], min_lesion_area=min_lesion_area)
                        processed.append(cleaned)
                    pred_binary = torch.stack(processed).cuda()
                else:
                    pred_binary = (pred_probs > 0.5).float()

                # Compute metrics using pred_logits for compatibility
                # We simulate logits from binary prediction
                pred_logits = torch.where(pred_binary > 0.5,
                    torch.tensor(10.0, device=pred_binary.device),
                    torch.tensor(-10.0, device=pred_binary.device))

                batch_metrics = calculate_segmentation_metrics(pred_logits, masks)
                for k in accumulated_metrics:
                    accumulated_metrics[k] += batch_metrics[k]

        # Average metrics
        for k in accumulated_metrics:
            accumulated_metrics[k] /= num_batches

        results[name] = accumulated_metrics

        print(f"{name:<20} | {'Lesion':<16} | {accumulated_metrics['lesion_dice']:<10.4f} | {accumulated_metrics['lesion_iou']:<10.4f} | {accumulated_metrics['lesion_precision']:<12.4f} | {accumulated_metrics['lesion_recall']:<12.4f}")
        print(f"{'':<20} | {'Background':<16} | {accumulated_metrics['background_dice']:<10.4f} | {accumulated_metrics['background_iou']:<10.4f} | {accumulated_metrics['background_precision']:<12.4f} | {accumulated_metrics['background_recall']:<12.4f}")
        print("-" * 90)

    return results

### 11. Main Training Pipeline\n\nUsing the improved pipeline:\n- BalancedNCCTDataset with patch extraction + weight maps\n- BalancedBatchSampler with positive oversampling\n- Enhanced augmentations\n- WeightedComboLoss (Focal Tversky + Lovász + Asymmetric BCE)\n- Two-phase LR + gradient clipping + EMA\n\nTrained models (3 best to save Kaggle compute):\n1. UNet_EffNet (efficientnet-b3)\n2. UNet_EffNet_B4 (efficientnet-b4, stronger encoder)\n3. AttentionUNet (resnet34 + scse)

In [ ]:
# Initialize balanced dataloaders
train_loader, val_loader, test_loader, train_dataset = get_balanced_dataloaders('dataset', batch_size=8)

# Create balanced batch sampler for training
balanced_sampler = BalancedBatchSampler(train_dataset, batch_size=8, min_pos=3)
balanced_train_loader = DataLoader(
    train_dataset,
    batch_sampler=balanced_sampler,
    num_workers=2
)

trained_models = {}
model_histories = {}

print("=" * 60)
print("Starting Enhanced Training Pipeline")
print("=" * 60)

#### Training Model 1: UNet_EffNet (EfficientNet-b3)

In [ ]:
name = "UNet_EffNet"
model = UNet_EffNet().cuda()
model = init_seg_head_bias(model, pos_weight=66.6)
m, h = train_model_weighted(
    model, balanced_train_loader, val_loader,
    epochs=50, model_name=name
)
trained_models[name] = m
model_histories[name] = h

print(f"\n=== Evaluating {name} (with TTA + Post-Processing) ===")
eval_model_advanced({name: m}, test_loader, use_tta=True, use_post_process=True)

#### Training Model 2: UNet_EffNet_B4 (EfficientNet-b4)

In [ ]:
name = "UNet_EffNet_B4"
model = UNet_EffNet_B4().cuda()
model = init_seg_head_bias(model, pos_weight=66.6)
m, h = train_model_weighted(
    model, balanced_train_loader, val_loader,
    epochs=50, model_name=name
)
trained_models[name] = m
model_histories[name] = h

print(f"\n=== Evaluating {name} (with TTA + Post-Processing) ===")
eval_model_advanced({name: m}, test_loader, use_tta=True, use_post_process=True)

#### Training Model 3: AttentionUNet (ResNet34 + scse attention)

In [ ]:
name = "AttentionUNet"
model = AttentionUNet().cuda()
model = init_seg_head_bias(model, pos_weight=66.6)
m, h = train_model_weighted(
    model, balanced_train_loader, val_loader,
    epochs=50, model_name=name
)
trained_models[name] = m
model_histories[name] = h

print(f"\n=== Evaluating {name} (with TTA + Post-Processing) ===")
eval_model_advanced({name: m}, test_loader, use_tta=True, use_post_process=True)

### 11.b. Save All Trained Model Weights

Save all trained model weights to `trained_models/` directory.
On Kaggle, download the generated zip file from the Output tab to persist weights for future sessions.

In [ ]:
def save_model_weights(models_dict, model_dir='trained_models'):
    """Save all trained model weights to a persistent directory."""
    os.makedirs(model_dir, exist_ok=True)
    print(f"Saving model weights to {model_dir}/ ...")
    for name, model in models_dict.items():
        path = os.path.join(model_dir, f"{name.lower().replace(' ', '_')}.pth")
        if hasattr(model, 'module'):  # DataParallel
            torch.save(model.module.state_dict(), path)
        else:
            torch.save(model.state_dict(), path)
        print(f"  \u2713 {name} -> {path}")
    print(f"\nAll weights saved to {model_dir}/")

def zip_models(model_dir='trained_models', zip_name='trained_models.zip'):
    """Create a zip file of all model weights for easy download."""
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(model_dir):
            for fname in files:
                if fname.endswith('.pth'):
                    file_path = os.path.join(root, fname)
                    zf.write(file_path, arcname=fname)
    size_mb = os.path.getsize(zip_name) / (1024*1024)
    print(f"Created {zip_name} ({size_mb:.2f} MB)")
    print("Download this file from Kaggle Output tab to persist weights.")
    return zip_name

# Save and zip
save_model_weights(trained_models)
zip_models()

if IS_KAGGLE:
    print("\nNEXT STEPS TO PERSIST WEIGHTS:")
    print("1. Go to the Output tab in Kaggle (right sidebar)")
    print("2. Find and download 'trained_models.zip'")
    print("3. Upload to a Kaggle Dataset for reuse")
    print("4. In future runs, use the 'load_model_weights' function from the dataset")


### 11.c. Load Pre-trained Weights (Optional - Skip Training)

If you have previously saved weights (e.g., from a Kaggle Dataset), load them here to skip retraining.
Use this when you want to run inference, visualization, or ensemble without retraining.

In [ ]:
def load_model_weights(models_dict, model_dir='trained_models'):
    """Load saved weights for all models."""
    loaded_any = False
    for name in list(models_dict.keys()):
        path = os.path.join(model_dir, f"{name.lower().replace(' ', '_')}.pth")
        if os.path.exists(path):
            models_dict[name].load_state_dict(torch.load(path, map_location='cuda'))
            print(f"  \u2713 Loaded {name} from {path}")
            loaded_any = True
        else:
            print(f"  \u2717 No saved weights found for {name}")
    if loaded_any:
        print("\nLoaded weights successfully. Models ready for inference/evaluation.")
    else:
        print("\nNo saved weights found. Train the models first.")
    return loaded_any

# Uncomment and modify the path below to load from Kaggle Dataset:
# LOAD_DIR = '/kaggle/input/YOUR-DATASET-NAME'
# load_model_weights(trained_models, model_dir=LOAD_DIR)

# To skip training and use pre-trained weights,
# comment out the training cells in Section 11
# and uncomment the loading code above.


### 12. Enhanced Visualization

Enhanced version of V5's visualization:
- Shows input, GT, and predictions for each model
- ADD: Post-processed prediction overlay
- ADD: Difference map (GT vs prediction) highlighted in red (FN) / green (FP)

In [ ]:
def visualize_all_models_enhanced(models_dict, test_loader, num_samples=3, use_tta=True):
    """
    Enhanced visualization with post-processed overlays and difference maps.
    """
    model_names = list(models_dict.keys())
    num_models = len(model_names)

    # Columns: Input, GT, then for each model: Prediction, Overlay, Diff
    cols_per_model = 3  # pred, overlay, diff
    total_cols = 2 + num_models * cols_per_model

    plt.figure(figsize=(5 * total_cols, 5 * num_samples))

    with torch.no_grad():
        for i in range(num_samples):
            images, masks, _ = next(iter(test_loader))
            images_gpu = images.cuda()

            # Plot Input
            plt.subplot(num_samples, total_cols, i * total_cols + 1)
            plt.imshow(images[0][0].cpu().numpy(), cmap='gray')
            plt.title("Input")
            plt.axis('off')

            # Plot Ground Truth
            plt.subplot(num_samples, total_cols, i * total_cols + 2)
            plt.imshow(masks[0][0].cpu().numpy(), cmap='gray')
            plt.title("GT")
            plt.axis('off')

            # Plot each model's prediction
            for j, name in enumerate(model_names):
                model = models_dict[name]
                model.eval()

                if use_tta:
                    pred_probs = apply_tta(model, images_gpu)
                else:
                    out = model(images_gpu)
                    if isinstance(out, list):
                        out = out[0]
                    pred_probs = torch.sigmoid(out)

                pred_binary = (pred_probs > 0.5).float()
                pred_np = pred_binary[0][0].cpu().numpy()
                gt_np = masks[0][0].cpu().numpy()

                col_offset = 3 + j * cols_per_model

                # Prediction
                plt.subplot(num_samples, total_cols, i * total_cols + col_offset)
                plt.imshow(pred_np, cmap='gray')
                plt.title(f"{name}")
                plt.axis('off')

                # Overlay (prediction on top of image)
                plt.subplot(num_samples, total_cols, i * total_cols + col_offset + 1)
                img_display = images[0][0].cpu().numpy()
                plt.imshow(img_display, cmap='gray', alpha=0.7)
                plt.imshow(pred_np, cmap='Reds', alpha=0.3)
                plt.title(f"{name} Overlay")
                plt.axis('off')

                # Difference map: Red=FN (missed), Green=FP (extra)
                plt.subplot(num_samples, total_cols, i * total_cols + col_offset + 2)
                diff_map = np.zeros((*pred_np.shape, 3))
                # False Negatives (GT=1, Pred=0) -> Red
                fn_mask = (gt_np > 0.5) & (pred_np < 0.5)
                diff_map[fn_mask] = [1.0, 0.0, 0.0]
                # False Positives (GT=0, Pred=1) -> Green
                fp_mask = (gt_np < 0.5) & (pred_np > 0.5)
                diff_map[fp_mask] = [0.0, 1.0, 0.0]
                # True Positives (GT=1, Pred=1) -> Yellow
                tp_mask = (gt_np > 0.5) & (pred_np > 0.5)
                diff_map[tp_mask] = [1.0, 1.0, 0.0]
                plt.imshow(diff_map)
                plt.title(f"{name} Diff (R=FN, G=FP)")
                plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Display enhanced visualization
visualize_all_models_enhanced(trained_models, test_loader, num_samples=2, use_tta=True)

### 13. Ensemble Prediction

Average predictions from all trained models with TTA + post-processing.
The ensemble typically outperforms individual models by reducing variance.

In [ ]:
def ensemble_predict(models_dict, image, use_tta=True, use_post_process=True, min_lesion_area=50):
    """
    Ensemble prediction: average probabilities from all models.
    """
    all_probs = []
    for name, model in models_dict.items():
        model.eval()
        with torch.no_grad():
            if use_tta:
                probs = apply_tta(model, image)
            else:
                out = model(image)
                if isinstance(out, list):
                    out = out[0]
                probs = torch.sigmoid(out)
        all_probs.append(probs)

    # Average probabilities
    ensemble_probs = torch.mean(torch.stack(all_probs), dim=0)

    # Post-process
    if use_post_process:
        ensemble_bin = (ensemble_probs > 0.5).float()
        processed = []
        for b in range(ensemble_bin.size(0)):
            cleaned = post_process_mask(ensemble_bin[b], min_lesion_area=min_lesion_area)
            processed.append(cleaned)
        ensemble_bin = torch.stack(processed).cuda()
        # Regenerate logits for metrics
        ensemble_logits = torch.where(ensemble_bin > 0.5,
            torch.tensor(10.0, device=ensemble_bin.device),
            torch.tensor(-10.0, device=ensemble_bin.device))
    else:
        ensemble_logits = ensemble_probs  # Will use default threshold

    return ensemble_logits, ensemble_probs


def eval_ensemble(models_dict, test_loader, use_tta=True, use_post_process=True, min_lesion_area=50):
    """
    Evaluate ensemble of models.
    """
    accumulated_metrics = {
        'lesion_dice': 0.0, 'lesion_iou': 0.0, 'lesion_precision': 0.0, 'lesion_recall': 0.0,
        'background_dice': 0.0, 'background_iou': 0.0, 'background_precision': 0.0, 'background_recall': 0.0
    }

    num_batches = len(test_loader)
    with torch.no_grad():
        for images, masks, _ in test_loader:
            images, masks = images.cuda(), masks.cuda()
            ensemble_logits, _ = ensemble_predict(
                models_dict, images,
                use_tta=use_tta, use_post_process=use_post_process,
                min_lesion_area=min_lesion_area
            )
            batch_metrics = calculate_segmentation_metrics(ensemble_logits, masks)
            for k in accumulated_metrics:
                accumulated_metrics[k] += batch_metrics[k]

    for k in accumulated_metrics:
        accumulated_metrics[k] /= num_batches

    return accumulated_metrics


# Compute and display ensemble results
print("\n" + "=" * 60)
print("ENSEMBLE PREDICTION RESULTS")
print("=" * 60)
print(f"Models in ensemble: {list(trained_models.keys())}")
print(f"\nEnsemble Metrics (TTA + Post-Processing):")
print("-" * 60)

ensemble_results = eval_ensemble(
    trained_models, test_loader,
    use_tta=True, use_post_process=True, min_lesion_area=50
)

print(f"  Lesion Dice Score:      {ensemble_results['lesion_dice']:.4f}")
print(f"  Lesion IoU (Jaccard):   {ensemble_results['lesion_iou']:.4f}")
print(f"  Lesion Precision:       {ensemble_results['lesion_precision']:.4f}")
print(f"  Lesion Recall:          {ensemble_results['lesion_recall']:.4f}")
print(f"  Background Dice:        {ensemble_results['background_dice']:.4f}")
print(f"  Background IoU:         {ensemble_results['background_iou']:.4f}")
print(f"  Background Precision:   {ensemble_results['background_precision']:.4f}")
print(f"  Background Recall:      {ensemble_results['background_recall']:.4f}")
print("=" * 60)